# RAW to RGB — Part 2: Debayering

The sensor records only one colour per pixel arranged in a **Bayer pattern**:

```
R G R G ...
G B G B ...
R G R G ...
...........
```

To obtain a full-colour image we need to **interpolate** the two missing colour values at each pixel.
This process is called **debayering** or **demosaicing**.

In this notebook we implement **bilinear debayering** step by step.

In [ ]:
import numpy as np              # needed for image arrays
import HdM as HdM               # a useful function I prepared for you to display images pixel per pixel on your monitor
import imageio.v3 as iio        # Reading images with OpenImageIO
from scipy.ndimage import zoom  # Useful function to scale images

linear2sRGB = lambda x: np.where(x > 0.0031308, 1.055 * np.clip(x, 0, None) ** (1/2.4) - 0.055, 12.92 * x)

data = np.load('20_RawBayerImage_BlackSubtracted.npz')
raw_img_black_subtracted = data['raw_img_black_subtracted']
black_noise_std      = data['black_noise_std']

print(f'Loaded image shape (Dimensions in Pixels): {raw_img_black_subtracted.shape}')

## Look closely at the Bayer pattern

Zoom into a small crop — you should clearly see the RGGB mosaic.

In [ ]:
stops_above_white = 2
crop = raw_img_black_subtracted[1000:1064, 1000:1064]

HdM.show( linear2sRGB( np.clip( np.repeat( np.repeat(crop, 8, axis=0), 8, axis=1) * 2**stops_above_white, 0, 1)))
print('64×64 crop — individual Bayer pixels visible')


## Split the Bayer plane into 4 sub-images

The RGGB Bayer pattern maps to four sub-sampled images:

| c1 | c3 | c1 | c3 | ... |
|---|---|---|---|---|
| c2 | c4 | c2 | c4 | ... |
| c1 | c3 | c1 | c3 | ... |
| c2 | c4 | c2 | c4 | ... |
| ... | ... | ... | ... | ... |


In Python array indexing rows and columns start at 0, so `0::2` selects every second element starting at 0 (even), `1::2` selects odd.

In [ ]:
c1 = raw_img_black_subtracted[ ___ , ___ ]  # Hinweis: Use your knowledge about indexing to only select the top-left sensel of each 2x2 block.
c2 = raw_img_black_subtracted[ ___ , ___ ]  # Hinweis: Use your knowledge about indexing to only select the bottom-left sensel of each 2x2 block.
c3 = raw_img_black_subtracted[ ___ , ___ ]  # Hinweis: Use your knowledge about indexing to only select the top-right sensel of each 2x2 block.
c4 = raw_img_black_subtracted[ ___ , ___ ]  # Hinweis: Use your knowledge about indexing to only select the bottom-right sensel of each 2x2 block.

# Stick together all four images
composite_img = np.vstack([np.hstack([c1,c2]),np.hstack([c3,c4])])

print('Four Bayer sub-images — which two are green? Which one represents red, which one blue?')
HdM.show(linear2sRGB(np.clip(zoom(composite_img, 0.2) * 2**stops_above_white, 0, 1)))
print('Can you guess the green channels?')

## Identify the colour channels

The two green channels should look the most similar to each other.
We compute the **sum of absolute differences** between all channel pairs to find them programmatically.

In [ ]:
def sad(a, b):
    """Sum of absolute differences."""
        return  # Hinweis: Calculate the sum of absolute differences

# Print the sum of absolute differences for each pair:
for pair in [('c1','c2'), ('c1','c3'), ('c1','c4'), ('c2','c3'), ('c2','c4'), ('c3','c4')]:
    channels = {'c1': c1, 'c2': c2, 'c3': c3, 'c4': c4}
    print(f'SAD({pair[0]}, {pair[1]}) = {sad(channels[pair[0]], channels[pair[1]]):.0f}')

## Simple 1:4 combine (Canon C300 MkI-style)

The quickest approach: average the two green channels, keep R and B as-is.
This gives a half-resolution colour image.

In [ ]:
r =   # Hinweis: Assign the red channel to variable "r"
g =   # Hinweis: Calculate "g" as the mean of both green channels
b =   # Hinweis: Assign the blue channel to variable "b"

rgb_small = np.stack([r, g, b], axis=2)

print('Half-resolution colour image (direct channel combine)')
HdM.show(linear2sRGB(np.clip( zoom( rgb_small, [0.2,0.2,1] ) * 2**stops_above_white, 0, 1)))
print('Does the color checker and the skin tones look about right? But why is the image tinted green?')

## Full-resolution bilinear debayering

To keep full resolution we **interpolate** each missing colour value from its neighbours.
For a RGGB pattern the bilinear interpolation kernels are:

**Red** (known at top-left of each 2×2 block):
```
1/4  1/2  1/4
1/2   1   1/2
1/4  1/2  1/4
```
applied only at non-R positions.

In practice the cleanest implementation uses **convolution with known-pixel masks**.
We follow the standard approach: convolve the full Bayer plane with colour-specific kernels
after masking out the wrong-colour pixels.

In [ ]:
# As a next step you will implement the most basic 1:1 debayering algorithm: Bilinear Interpolation
# But before we start lets's bring all raw images to the same RGB pattern. Lets make the first pixel a blue sensitive sensel:

# If your blue channel is c1 uncomment and run the following command:
# raw = raw_img_black_subtracted[0:, 0:] # This actually does nothing

# If your blue channel is c2 uncomment and run the following command:
# raw = raw_img_black_subtracted[1:-1, 0:] # This crops the first and last row

# If your blue channel is c3 uncomment and run the following command:
raw = raw_img_black_subtracted[0:, 1:-1] # This crops the first and last column

# If your blue channel is c4 uncomment and run the following command:
# raw = raw_img_black_subtracted[1:-1, 1:-1] # This crops the first and last row and column

rgb_small = np.stack([raw[1::2,1::2], (raw[1::2,0::2]+raw[0::2,1::2])/2, raw[0::2,0::2]], axis=2)

print('Half-resolution colour image (direct channel combine). Should look exactly the same as your c1, c2, c3, c4 combination above.')
HdM.show(linear2sRGB(np.clip( zoom( rgb_small, [0.2,0.2,1] ) * 2**stops_above_white, 0, 1)))


In [ ]:

# Lets first initialize three image planes full of zeros:
r = np.zeros((raw.shape[0]-2, raw.shape[1]-2))
g = np.zeros((raw.shape[0]-2, raw.shape[1]-2))
b = np.zeros((raw.shape[0]-2, raw.shape[1]-2))

# The new image planes are one sensel smaller compared to raw at each border, left, right, top bottom.
# As a result they starts with a red sensitive pixel. Hence we can directly write the red sensel values into the red image plane:
r[0::2,0::2] = raw[1:-1:2,1:-1:2]

# Now it's your turn. Calculate the other three red pixel values: 
r[1::2,0::2] = (raw[1:-1:2, 1:-1:2] + raw[3:  :2, 1:-1:2]) / 2
r[0::2,1::2] = (raw[1:-1:2, 1:-1:2] + raw[1:-1:2, 3::2  ]) / 2
r[1::2,1::2] = (raw[1:-1:2, 1:-1:2] + raw[3:  :2, 1:-1:2] + raw[1:-1:2, 3::2] + raw[3:  :2, 3:  :2]) / 4

print("This is your full resolution reconstruction of the red color channel")
HdM.show(linear2sRGB(np.clip(r * 2**stops_above_white, 0, 1)))

In [ ]:

# Next reconsrtuct the green channel by bilinear interploation:
g[0::2,0::2] = raw[1:-1:2,1:-1:2]
g[1::2,0::2] = (raw[1:-1:2, 1:-1:2] + raw[3:  :2, 1:-1:2]) / 2
g[0::2,1::2] = (raw[1:-1:2, 1:-1:2] + raw[1:-1:2, 3::2  ]) / 2
g[1::2,1::2] = (raw[1:-1:2, 1:-1:2] + raw[3:  :2, 1:-1:2] + raw[1:-1:2, 3::2] + raw[3:  :2, 3:  :2]) / 4

print("This is your full resolution reconstruction of the green color channel")
HdM.show(linear2sRGB(np.clip(g * 2**stops_above_white, 0, 1)))


In [ ]:
# Next reconsrtuct the blue channel by bilinear interploation:
b[0::2,0::2] = raw[1:-1:2,1:-1:2]
b[1::2,0::2] = (raw[1:-1:2, 1:-1:2] + raw[3:  :2, 1:-1:2]) / 2
b[0::2,1::2] = (raw[1:-1:2, 1:-1:2] + raw[1:-1:2, 3::2  ]) / 2
b[1::2,1::2] = (raw[1:-1:2, 1:-1:2] + raw[3:  :2, 1:-1:2] + raw[1:-1:2, 3::2] + raw[3:  :2, 3:  :2]) / 4

print("This is your full resolution reconstruction of the blue color channel")
HdM.show(linear2sRGB(np.clip(b * 2**stops_above_white, 0, 1)))

In [ ]:
# Finally, we can combine r,g and b to rgb and display it:

rgb = np.stack([r, g, b], axis=2)

HdM.show(linear2sRGB(np.clip(rgb * 2**stops_above_white, 0, 1)))

# You made it. Congratulations!


## Compare: direct combine vs. bilinear interpolation

We crop a small region to compare resolution and edge sharpness.

In [ ]:
vo = 2260 // 2   # vertical offset (half-res coordinates for rgb_small)
ho = 2780 // 2   # horizontal offset (half-res coordinates for rgb_small)
crop_size = 100
stops = 1

crop_small = rgb_small[vo:vo+crop_size, ho:ho+crop_size, :]
# Upsample for display by nearest-neighbour (zoom=2)
crop_small_up = np.repeat(np.repeat(crop_small, 2, axis=0), 2, axis=1)

crop_full = rgb[vo*2:(vo+crop_size)*2, ho*2:(ho+crop_size)*2, :]

print('4:1 combine (half-res) + upscale on the left compared to bilinear interpolation (full-res) on the right')
HdM.show(np.hstack([crop_small_up,crop_full]))

print('Edges on the right side show zipper artifacts — fixed in next notebook with edge-directed debayering')


## Save result

In [ ]:
np.savez('21_RGB_BilinearDebayer.npz',
         rgb=rgb,
         img_black_subtracted_for_debayering=bayer,
         black_noise_std=black_noise_std)

print('Saved: 21_RGB_BilinearDebayer.npz')